# Lab 6 - LSTM và GRU dự đoán Vietlott 6/55

Notebook này được viết theo cấu trúc Kaggle và dùng file `result.csv` để xây dựng hai mô hình **LSTM** và **GRU** dự đoán kết quả Vietlott 6/55.

**Lưu ý học thuật:** xổ số là quá trình gần như ngẫu nhiên, nên mục tiêu chính của bài là rèn luyện quy trình xử lý dữ liệu chuỗi thời gian, thiết kế mô hình hồi quy/gated recurrent và đánh giá kết quả. Không nên hiểu kết quả dự đoán như một khuyến nghị chơi xổ số.

## 1. Thiết lập môi trường Kaggle

- Kaggle thường đã có sẵn `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `torch`.
- Notebook tự chọn `cuda` nếu có GPU, còn không sẽ chạy CPU bình thường.
- Đường dẫn dữ liệu được tự dò trong `/kaggle/input`, hoặc fallback về thư mục local khi chạy trên máy cá nhân.

In [ ]:
# Nếu Kaggle thiếu thư viện, có thể mở comment dòng dưới.
# !pip install -q torch numpy pandas matplotlib scikit-learn

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import hamming_loss

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị đang sử dụng:", device)

## 2. Đọc dữ liệu

Trên Kaggle, hãy upload `result.csv` thành một Dataset rồi attach vào Notebook. Cell dưới đây sẽ tự tìm file trong `/kaggle/input/**/result.csv`.

In [ ]:
def find_data_path(filename="result.csv"):
    candidates = []
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(kaggle_root.rglob(filename))

    local_candidates = [
        Path("result.csv"),
        Path("/home/yennguyen/DeepLearning/lab06/result.csv"),
        Path("./lab06/result.csv"),
    ]
    candidates.extend([p for p in local_candidates if p.exists()])

    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy result.csv. Trên Kaggle hãy attach dataset chứa file result.csv."
        )
    return candidates[0]

DATA_PATH = find_data_path("result.csv")
print("Đường dẫn dữ liệu:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Kích thước dữ liệu:", df.shape)
print("Các cột:", list(df.columns))
display(df.info())
display(df.describe())

## 3. Tiền xử lý dữ liệu

CSV có cột `Ngày`, các cột `Số 1` đến `Số 6` là 6 số chính của Vietlott 6/55. Cột `Số 7` được giữ làm số đặc biệt/tham khảo, nhưng mô hình chính dự đoán 6 số chính.

Mỗi kỳ quay được mã hóa thành vector nhị phân 55 chiều:

- vị trí `i-1 = 1` nếu số `i` xuất hiện trong kỳ quay;
- còn lại bằng `0`.

Sau đó dùng cửa sổ trượt: `sequence_length` kỳ trước để dự đoán kỳ kế tiếp.

In [ ]:
DATE_COL = "Ngày"
MAIN_NUMBER_COLS = ["Số 1", "Số 2", "Số 3", "Số 4", "Số 5", "Số 6"]
BONUS_COL = "Số 7" if "Số 7" in df.columns else None
N_NUMBERS = 55
N_PICK = 6

# Chuẩn hóa tên/cột và kiểu dữ liệu
data = df.copy()
data[DATE_COL] = pd.to_datetime(data[DATE_COL], dayfirst=True, errors="coerce")
for col in MAIN_NUMBER_COLS + ([BONUS_COL] if BONUS_COL else []):
    data[col] = pd.to_numeric(data[col], errors="coerce")

data = data.dropna(subset=[DATE_COL] + MAIN_NUMBER_COLS).sort_values(DATE_COL).reset_index(drop=True)
for col in MAIN_NUMBER_COLS:
    data[col] = data[col].astype(int)

print("Số kỳ quay sau xử lý:", len(data))
print("Từ ngày", data[DATE_COL].min().date(), "đến", data[DATE_COL].max().date())
display(data.head())
display(data.tail())

# Kiểm tra miền giá trị
for col in MAIN_NUMBER_COLS:
    mn, mx = data[col].min(), data[col].max()
    print(f"{col}: min={mn}, max={mx}")

In [ ]:
def draw_to_multihot(numbers, n_numbers=N_NUMBERS):
    vec = np.zeros(n_numbers, dtype=np.float32)
    for n in numbers:
        if 1 <= int(n) <= n_numbers:
            vec[int(n) - 1] = 1.0
    return vec

multi_hot = np.vstack([
    draw_to_multihot(row[MAIN_NUMBER_COLS].values)
    for _, row in data.iterrows()
])

print("Shape multi-hot:", multi_hot.shape)
print("Tổng số bit 1 mỗi kỳ, 5 dòng đầu:", multi_hot[:5].sum(axis=1))

## 4. Khám phá dữ liệu

Ta xem tần suất xuất hiện của từng số và khoảng cách giữa các kỳ quay. Với xổ số, tần suất có thể dao động trong lịch sử nhưng không đảm bảo khả năng dự đoán tương lai.

In [ ]:
number_freq = multi_hot.sum(axis=0)
freq_df = pd.DataFrame({
    "number": np.arange(1, N_NUMBERS + 1),
    "frequency": number_freq.astype(int),
    "rate": number_freq / len(data),
}).sort_values("frequency", ascending=False)

display(freq_df.head(10))
display(freq_df.tail(10))

fig, axes = plt.subplots(2, 1, figsize=(15, 8))
axes[0].plot(data[DATE_COL], data[MAIN_NUMBER_COLS], marker=".", linewidth=0.6, alpha=0.8)
axes[0].set_title("Chuỗi các số chính theo thời gian")
axes[0].set_xlabel("Ngày")
axes[0].set_ylabel("Giá trị số")
axes[0].legend(MAIN_NUMBER_COLS, ncol=6, fontsize=9)

axes[1].bar(freq_df.sort_values("number")["number"], freq_df.sort_values("number")["frequency"], color="#2E86AB")
axes[1].axhline(number_freq.mean(), color="#D1495B", linestyle="--", label="Tần suất trung bình")
axes[1].set_title("Tần suất xuất hiện của từng số 1-55")
axes[1].set_xlabel("Số")
axes[1].set_ylabel("Số lần xuất hiện")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Tạo dữ liệu chuỗi thời gian

Không shuffle khi chia train/validation/test vì đây là dữ liệu chuỗi thời gian. Nếu shuffle, mô hình có thể học từ các kỳ tương lai rồi đánh giá trên quá khứ, gây **data leakage**.

In [ ]:
def create_sequences(values, sequence_length=20):
    X, y = [], []
    for i in range(len(values) - sequence_length):
        X.append(values[i:i + sequence_length])
        y.append(values[i + sequence_length])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

def split_time_series(X, y, train_ratio=0.70, val_ratio=0.15):
    n = len(X)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return (
        X[:train_end], y[:train_end],
        X[train_end:val_end], y[train_end:val_end],
        X[val_end:], y[val_end:],
    )

SEQUENCE_LENGTH = 20
X, y = create_sequences(multi_hot, sequence_length=SEQUENCE_LENGTH)
X_train, y_train, X_val, y_val, X_test, y_test = split_time_series(X, y)

print("X shape:", X.shape, "y shape:", y.shape)
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

In [ ]:
class VietlottSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32
train_loader = DataLoader(VietlottSequenceDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(VietlottSequenceDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(VietlottSequenceDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

## 6. Định nghĩa mô hình LSTM và GRU

Mô hình nhận input `(batch, sequence_length, 55)` và xuất ra 55 logits. Sau sigmoid, mỗi logit là xác suất số tương ứng xuất hiện trong kỳ tiếp theo.

In [ ]:
class RecurrentLotteryModel(nn.Module):
    def __init__(self, model_type="LSTM", input_size=55, hidden_size=96, num_layers=2, dropout=0.2, output_size=55):
        super().__init__()
        self.model_type = model_type.upper()
        rnn_cls = {"LSTM": nn.LSTM, "GRU": nn.GRU}[self.model_type]
        self.rnn = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, _ = self.rnn(x)
        last_output = output[:, -1, :]
        last_output = self.norm(last_output)
        last_output = self.dropout(last_output)
        return self.fc(last_output)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

for model_type in ["LSTM", "GRU"]:
    model = RecurrentLotteryModel(model_type=model_type)
    print(f"{model_type}: {model.count_parameters():,} tham số")

## 7. Hàm huấn luyện và đánh giá

Loss chính là `BCEWithLogitsLoss` vì đây là bài toán multi-label 55 nhãn. Metric chính:

- `avg_hits@6`: số lượng số đúng trung bình khi lấy top 6 dự đoán;
- `exact_match_rate`: tỷ lệ dự đoán đúng toàn bộ 6 số, thường rất thấp với xổ số;
- `hamming_loss`: lỗi nhị phân trung bình trên 55 nhãn.

In [ ]:
def top_k_multihot(probabilities, k=N_PICK):
    top_idx = np.argsort(-probabilities, axis=1)[:, :k]
    pred = np.zeros_like(probabilities, dtype=np.float32)
    rows = np.arange(probabilities.shape[0])[:, None]
    pred[rows, top_idx] = 1.0
    return pred

def compute_metrics_from_probs(probs, y_true, k=N_PICK):
    pred = top_k_multihot(probs, k=k)
    hits = (pred * y_true).sum(axis=1)
    exact = (hits == k).mean()
    return {
        "avg_hits@6": float(hits.mean()),
        "max_hits@6": int(hits.max()) if len(hits) else 0,
        "exact_match_rate": float(exact),
        "hamming_loss": float(hamming_loss(y_true.astype(int), pred.astype(int))),
    }

def evaluate_model(model, loader, criterion=None):
    model.eval()
    all_probs, all_targets = [], []
    total_loss, n_batches = 0.0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device)
            by = by.to(device)
            logits = model(bx)
            if criterion is not None:
                total_loss += criterion(logits, by).item()
                n_batches += 1
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_probs.append(probs)
            all_targets.append(by.detach().cpu().numpy())
    probs = np.vstack(all_probs)
    targets = np.vstack(all_targets)
    metrics = compute_metrics_from_probs(probs, targets)
    metrics["loss"] = total_loss / max(n_batches, 1) if criterion is not None else np.nan
    return metrics, probs, targets

def train_model(model, train_loader, val_loader, epochs=80, lr=1e-3, weight_decay=1e-4, grad_clip=1.0, patience=15):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "val_hits": []}
    best_state = None
    best_val_loss = float("inf")
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, n_batches = 0.0, 0
        for bx, by in train_loader:
            bx = bx.to(device)
            by = by.to(device)
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1

        train_loss = running_loss / max(n_batches, 1)
        val_metrics, _, _ = evaluate_model(model, val_loader, criterion)
        val_loss = val_metrics["loss"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_hits"].append(val_metrics["avg_hits@6"])

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | train_loss={train_loss:.4f} "
                f"| val_loss={val_loss:.4f} | val_hits@6={val_metrics['avg_hits@6']:.3f}"
            )

        if wait >= patience:
            print(f"Early stopping tại epoch {epoch}. Best val_loss={best_val_loss:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

## 8. Baseline tần suất

Baseline đơn giản: luôn dự đoán 6 số xuất hiện nhiều nhất trong tập train. Mô hình LSTM/GRU nên được so với baseline này thay vì chỉ nhìn loss.

In [ ]:
train_freq = y_train.sum(axis=0)
baseline_numbers = np.argsort(-train_freq)[:N_PICK] + 1
print("6 số baseline theo tần suất train:", sorted(baseline_numbers.tolist()))

baseline_probs = np.tile(train_freq / len(y_train), (len(y_test), 1))
baseline_metrics = compute_metrics_from_probs(baseline_probs, y_test)
baseline_metrics

## 9. Huấn luyện LSTM

In [ ]:
set_seed(SEED)
lstm_model = RecurrentLotteryModel(
    model_type="LSTM",
    input_size=N_NUMBERS,
    hidden_size=96,
    num_layers=2,
    dropout=0.25,
    output_size=N_NUMBERS,
)
print(lstm_model)
lstm_model, lstm_history = train_model(
    lstm_model, train_loader, val_loader,
    epochs=80, lr=1e-3, weight_decay=1e-4, grad_clip=1.0, patience=15,
)

## 10. Huấn luyện GRU

In [ ]:
set_seed(SEED)
gru_model = RecurrentLotteryModel(
    model_type="GRU",
    input_size=N_NUMBERS,
    hidden_size=96,
    num_layers=2,
    dropout=0.25,
    output_size=N_NUMBERS,
)
print(gru_model)
gru_model, gru_history = train_model(
    gru_model, train_loader, val_loader,
    epochs=80, lr=1e-3, weight_decay=1e-4, grad_clip=1.0, patience=15,
)

## 11. Learning curves

In [ ]:
def plot_histories(histories):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, hist in histories.items():
        axes[0].plot(hist["train_loss"], label=f"{name} train")
        axes[0].plot(hist["val_loss"], label=f"{name} val")
        axes[1].plot(hist["val_hits"], label=name)

    axes[0].set_title("Learning curves - BCE loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].set_title("Validation avg hits@6")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Số dự đoán đúng trung bình")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_histories({"LSTM": lstm_history, "GRU": gru_history})

## 12. Đánh giá trên tập test

In [ ]:
criterion = nn.BCEWithLogitsLoss()
lstm_test_metrics, lstm_probs, test_targets = evaluate_model(lstm_model, test_loader, criterion)
gru_test_metrics, gru_probs, _ = evaluate_model(gru_model, test_loader, criterion)

results_df = pd.DataFrame([
    {"model": "Frequency baseline", **baseline_metrics, "loss": np.nan},
    {"model": "LSTM", **lstm_test_metrics},
    {"model": "GRU", **gru_test_metrics},
])
display(results_df)

In [ ]:
def numbers_from_multihot(vec):
    return (np.where(vec > 0.5)[0] + 1).tolist()

def numbers_from_probs(probs, k=N_PICK):
    return sorted((np.argsort(-probs)[:k] + 1).tolist())

compare_rows = []
for i in range(min(12, len(test_targets))):
    draw_index = len(data) - len(test_targets) + i
    compare_rows.append({
        "date": data.iloc[draw_index][DATE_COL].date(),
        "actual": sorted(numbers_from_multihot(test_targets[i])),
        "LSTM_pred": numbers_from_probs(lstm_probs[i]),
        "LSTM_hits": len(set(numbers_from_multihot(test_targets[i])) & set(numbers_from_probs(lstm_probs[i]))),
        "GRU_pred": numbers_from_probs(gru_probs[i]),
        "GRU_hits": len(set(numbers_from_multihot(test_targets[i])) & set(numbers_from_probs(gru_probs[i]))),
    })

prediction_examples = pd.DataFrame(compare_rows)
display(prediction_examples)

## 13. Biểu đồ dự đoán

Vì output là tập 6 số, biểu đồ dưới đây thể hiện `hits@6` theo từng mẫu test: mỗi điểm là số lượng số dự đoán đúng trong kỳ đó.

In [ ]:
def hits_series(probs, targets):
    pred = top_k_multihot(probs, k=N_PICK)
    return (pred * targets).sum(axis=1)

lstm_hits = hits_series(lstm_probs, test_targets)
gru_hits = hits_series(gru_probs, test_targets)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(lstm_hits, marker="o", linewidth=1, label="LSTM")
axes[0].plot(gru_hits, marker="s", linewidth=1, label="GRU")
axes[0].set_title("Hits@6 theo từng mẫu test")
axes[0].set_xlabel("Thứ tự mẫu test")
axes[0].set_ylabel("Số dự đoán đúng")
axes[0].set_ylim(-0.1, 6.1)
axes[0].legend()

axes[1].hist([lstm_hits, gru_hits], bins=np.arange(-0.5, 7.5, 1), label=["LSTM", "GRU"], alpha=0.75)
axes[1].set_title("Phân phối Hits@6")
axes[1].set_xlabel("Số dự đoán đúng")
axes[1].set_ylabel("Số mẫu")
axes[1].legend()
plt.tight_layout()
plt.show()

## 14. Dự đoán kỳ tiếp theo

Cell này dùng toàn bộ `sequence_length` kỳ gần nhất để dự đoán 6 số có xác suất cao nhất cho kỳ tiếp theo.

In [ ]:
def predict_next_draw(model, recent_sequence):
    model.eval()
    x = torch.tensor(recent_sequence[None, :, :], dtype=torch.float32).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(x)).cpu().numpy()[0]
    top_numbers = numbers_from_probs(probs, k=N_PICK)
    top_table = pd.DataFrame({
        "number": np.arange(1, N_NUMBERS + 1),
        "probability": probs,
    }).sort_values("probability", ascending=False).head(12)
    return top_numbers, top_table

recent_sequence = multi_hot[-SEQUENCE_LENGTH:]
lstm_next, lstm_top_table = predict_next_draw(lstm_model, recent_sequence)
gru_next, gru_top_table = predict_next_draw(gru_model, recent_sequence)

print("Ngày dữ liệu cuối:", data[DATE_COL].max().date())
print("LSTM dự đoán top 6:", lstm_next)
print("GRU  dự đoán top 6:", gru_next)
print("\nTop 12 xác suất LSTM")
display(lstm_top_table)
print("\nTop 12 xác suất GRU")
display(gru_top_table)

## 15. Thí nghiệm siêu tham số ngắn

Để notebook chạy được trên CPU/Kaggle nhanh hơn, phần này dùng số epoch nhỏ. Khi nộp báo cáo, có thể tăng `epochs` để kết quả ổn định hơn.

In [ ]:
def run_quick_experiment(sequence_lengths=(10, 20, 30), hidden_sizes=(64, 96), model_types=("LSTM", "GRU"), epochs=25):
    rows = []
    for seq_len in sequence_lengths:
        X_exp, y_exp = create_sequences(multi_hot, sequence_length=seq_len)
        Xtr, ytr, Xv, yv, Xte, yte = split_time_series(X_exp, y_exp)
        tr_loader = DataLoader(VietlottSequenceDataset(Xtr, ytr), batch_size=BATCH_SIZE, shuffle=False)
        v_loader = DataLoader(VietlottSequenceDataset(Xv, yv), batch_size=BATCH_SIZE, shuffle=False)
        te_loader = DataLoader(VietlottSequenceDataset(Xte, yte), batch_size=BATCH_SIZE, shuffle=False)

        for hidden in hidden_sizes:
            for model_type in model_types:
                print(f"\nThử nghiệm: {model_type}, seq={seq_len}, hidden={hidden}")
                set_seed(SEED)
                model = RecurrentLotteryModel(model_type=model_type, hidden_size=hidden, num_layers=2, dropout=0.2)
                model, hist = train_model(
                    model, tr_loader, v_loader,
                    epochs=epochs, lr=1e-3, weight_decay=1e-4, grad_clip=1.0, patience=8,
                )
                metrics, _, _ = evaluate_model(model, te_loader, nn.BCEWithLogitsLoss())
                rows.append({
                    "model": model_type,
                    "sequence_length": seq_len,
                    "hidden_size": hidden,
                    **metrics,
                    "best_val_loss": min(hist["val_loss"]),
                })
    return pd.DataFrame(rows).sort_values(["avg_hits@6", "loss"], ascending=[False, True])

experiment_df = run_quick_experiment(epochs=20)
display(experiment_df)

## 16. Thực hành các kỹ thuật cải thiện mô hình

Phần này bám sát yêu cầu nâng cao của bài thực hành: thử gradient clipping, dropout, weight decay, khởi tạo trọng số, LayerNorm và early stopping. Để phù hợp yêu cầu đề tài Vietlott, các thí nghiệm được áp dụng trên LSTM/GRU với dữ liệu `result.csv` thay vì dữ liệu thời tiết/chứng khoán.

Các thí nghiệm đặt số epoch vừa phải để chạy được trên Kaggle CPU. Khi cần báo cáo đẹp hơn, có thể tăng `epochs` từ 20-30 lên 80-150.

### 16.1 Gradient clipping

Mục tiêu: quan sát việc giới hạn gradient norm giúp training ổn định hơn khi learning rate lớn.

In [ ]:
def train_with_clipping_experiment(model_type="GRU", clip_values=(None, 5.0, 1.0, 0.5), epochs=30, lr=0.004):
    criterion = nn.BCEWithLogitsLoss()
    results = {}

    for clip in clip_values:
        label = "No clipping" if clip is None else f"clip={clip}"
        print(f"\n--- {model_type} | {label} ---")
        set_seed(SEED)
        model = RecurrentLotteryModel(model_type=model_type, hidden_size=96, num_layers=2, dropout=0.2).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        history = {"train_loss": [], "val_loss": [], "grad_norms": []}

        for epoch in range(epochs):
            model.train()
            epoch_loss, epoch_grad_norm, nb = 0.0, 0.0, 0
            for bx, by in train_loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()

                total_norm = 0.0
                for param in model.parameters():
                    if param.grad is not None:
                        total_norm += param.grad.detach().data.norm(2).item() ** 2
                total_norm = total_norm ** 0.5
                epoch_grad_norm += total_norm

                if clip is not None:
                    nn.utils.clip_grad_norm_(model.parameters(), clip)
                optimizer.step()
                epoch_loss += loss.item()
                nb += 1

            val_metrics, _, _ = evaluate_model(model, val_loader, criterion)
            history["train_loss"].append(epoch_loss / max(nb, 1))
            history["val_loss"].append(val_metrics["loss"])
            history["grad_norms"].append(epoch_grad_norm / max(nb, 1))

        results[label] = history
        print(f"Val loss cuối: {history['val_loss'][-1]:.4f}")
    return results

clip_results = train_with_clipping_experiment(model_type="GRU", epochs=25)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, history in clip_results.items():
    axes[0].plot(history["val_loss"], label=label)
    axes[1].plot(history["grad_norms"], label=label)
axes[0].set_title("Val Loss theo Gradient Clipping")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[1].set_title("Gradient Norm trước khi clip")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Norm")
axes[1].set_yscale("log")
for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét cần điền sau khi chạy:** so sánh đường `val_loss` và `grad_norms`. Nếu learning rate lớn làm gradient dao động mạnh, các cấu hình `clip=1.0` hoặc `clip=0.5` thường ổn định hơn. Nếu clip quá nhỏ, mô hình học chậm vì gradient bị co quá mạnh.

### 16.2 Dropout

Mục tiêu: đánh giá dropout có giúp giảm overfitting hay không bằng cách so sánh train loss và val loss.

In [ ]:
def dropout_experiment(dropout_rates=(0.0, 0.1, 0.2, 0.3, 0.5), epochs=30):
    results = {}
    for dr in dropout_rates:
        print(f"\n--- LSTM | dropout={dr} ---")
        set_seed(SEED)
        model = RecurrentLotteryModel(model_type="LSTM", hidden_size=128, num_layers=3, dropout=dr).to(device)
        model, history = train_model(
            model, train_loader, val_loader,
            epochs=epochs, lr=1e-3, weight_decay=1e-4, grad_clip=1.0, patience=10,
        )
        results[dr] = history
        gap = abs(history["train_loss"][-1] - history["val_loss"][-1])
        print(f"Train: {history['train_loss'][-1]:.4f} | Val: {history['val_loss'][-1]:.4f} | Gap: {gap:.4f}")
    return results

dropout_results = dropout_experiment(epochs=25)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for dr, history in dropout_results.items():
    axes[0].plot(history["train_loss"], label=f"drop={dr}")
    axes[1].plot(history["val_loss"], label=f"drop={dr}")
axes[0].set_title("Train Loss theo Dropout")
axes[1].set_title("Val Loss theo Dropout")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét cần điền sau khi chạy:** dropout thấp có thể chưa giảm overfitting rõ; dropout quá cao như `0.5` có thể làm train loss cao và gây underfitting. Chọn dropout dựa trên val loss thấp và khoảng cách train-val không quá lớn.

### 16.3 Weight decay

Mục tiêu: kiểm tra regularization L2 thông qua tham số `weight_decay` của optimizer.

In [ ]:
def weight_decay_experiment(weight_decays=(0, 1e-5, 1e-4, 1e-3, 1e-2), epochs=30):
    results = {}
    rows = []
    for wd in weight_decays:
        print(f"\n--- LSTM | weight_decay={wd} ---")
        set_seed(SEED)
        model = RecurrentLotteryModel(model_type="LSTM", hidden_size=96, num_layers=2, dropout=0.2).to(device)
        model, history = train_model(
            model, train_loader, val_loader,
            epochs=epochs, lr=1e-3, weight_decay=wd, grad_clip=1.0, patience=10,
        )
        metrics, _, _ = evaluate_model(model, val_loader, nn.BCEWithLogitsLoss())
        results[wd] = history
        rows.append({"weight_decay": wd, "final_val_loss": history["val_loss"][-1], "val_hits@6": metrics["avg_hits@6"]})
    return results, pd.DataFrame(rows)

wd_results, wd_table = weight_decay_experiment(epochs=25)
display(wd_table)

fig, ax = plt.subplots(figsize=(10, 5))
for wd, history in wd_results.items():
    label = "No decay" if wd == 0 else f"wd={wd}"
    ax.plot(history["val_loss"], label=label)
ax.set_title("Val Loss theo Weight Decay")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét cần điền sau khi chạy:** weight decay quá lớn như `1e-2` có thể ép trọng số quá nhỏ, làm mô hình đơn giản quá mức và underfitting. Giá trị vừa phải như `1e-5` hoặc `1e-4` thường an toàn hơn.

### 16.4 Khởi tạo trọng số

Mục tiêu: so sánh khởi tạo mặc định, Xavier, Orthogonal và Zeros. Với mạng hồi quy, Orthogonal thường giúp gradient ổn định hơn; Zeros thường rất tệ vì symmetry problem.

In [ ]:
def init_recurrent_weights(model, method="default"):
    for name, param in model.named_parameters():
        if "weight_ih" in name or "weight_hh" in name:
            if method == "xavier":
                nn.init.xavier_uniform_(param)
            elif method == "orthogonal":
                nn.init.orthogonal_(param)
            elif method == "zeros":
                nn.init.zeros_(param)
            elif method == "default":
                pass
        elif "bias" in name:
            nn.init.zeros_(param)
            # Với LSTM, đặt forget gate bias = 1 để mô hình dễ giữ thông tin ban đầu hơn.
            if method != "zeros" and "rnn" in name and param.numel() % 4 == 0:
                n = param.size(0)
                param.data[n // 4:n // 2].fill_(1.0)

def init_experiment(methods=("default", "xavier", "orthogonal", "zeros"), epochs=30):
    results = {}
    rows = []
    for method in methods:
        print(f"\n--- LSTM | init={method} ---")
        set_seed(SEED)
        model = RecurrentLotteryModel(model_type="LSTM", hidden_size=96, num_layers=2, dropout=0.2).to(device)
        init_recurrent_weights(model, method=method)
        model, history = train_model(
            model, train_loader, val_loader,
            epochs=epochs, lr=1e-3, weight_decay=1e-4, grad_clip=1.0, patience=10,
        )
        results[method] = history
        rows.append({"init_method": method, "final_val_loss": history["val_loss"][-1], "best_val_loss": min(history["val_loss"])})
    return results, pd.DataFrame(rows)

init_results, init_table = init_experiment(epochs=25)
display(init_table)

fig, ax = plt.subplots(figsize=(10, 5))
for method, history in init_results.items():
    ax.plot(history["val_loss"], label=method)
ax.set_title("Val Loss theo phương pháp khởi tạo")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét cần điền sau khi chạy:** `zeros` thường cho kết quả kém vì các neuron bắt đầu giống nhau và học các đặc trưng giống nhau. `orthogonal` thường phù hợp với recurrent weights vì giúp truyền gradient ổn định hơn qua thời gian.

### 16.5 LayerNorm

Mục tiêu: so sánh LSTM/GRU có và không có LayerNorm. Trong notebook chính, mô hình đang dùng LayerNorm; cell dưới tạo phiên bản không LayerNorm để đối chiếu.

In [ ]:
class RecurrentLotteryModelNoNorm(nn.Module):
    def __init__(self, model_type="LSTM", input_size=55, hidden_size=96, num_layers=2, dropout=0.2, output_size=55):
        super().__init__()
        self.model_type = model_type.upper()
        rnn_cls = {"LSTM": nn.LSTM, "GRU": nn.GRU}[self.model_type]
        self.rnn = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, _ = self.rnn(x)
        last_output = self.dropout(output[:, -1, :])
        return self.fc(last_output)

set_seed(SEED)
model_no_ln = RecurrentLotteryModelNoNorm(model_type="LSTM", hidden_size=128, num_layers=3, dropout=0.2).to(device)
model_no_ln, history_no_ln = train_model(
    model_no_ln, train_loader, val_loader,
    epochs=30, lr=0.002, weight_decay=1e-4, grad_clip=1.0, patience=10,
)

set_seed(SEED)
model_ln = RecurrentLotteryModel(model_type="LSTM", hidden_size=128, num_layers=3, dropout=0.2).to(device)
model_ln, history_ln = train_model(
    model_ln, train_loader, val_loader,
    epochs=30, lr=0.002, weight_decay=1e-4, grad_clip=1.0, patience=10,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history_no_ln["val_loss"], label="LSTM không LayerNorm")
ax.plot(history_ln["val_loss"], label="LSTM + LayerNorm")
ax.set_title("Ảnh hưởng của LayerNorm")
ax.set_xlabel("Epoch")
ax.set_ylabel("Val Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét cần điền sau khi chạy:** nếu đường val loss của mô hình có LayerNorm mượt hơn hoặc thấp hơn, LayerNorm đang giúp training ổn định. Nếu không cải thiện rõ, có thể do dữ liệu xổ số gần ngẫu nhiên hoặc cấu hình chưa đủ nhạy.

### 16.6 Early stopping và patience

Mục tiêu: thử các giá trị patience khác nhau. Patience quá nhỏ có thể dừng khi mô hình chưa kịp học; patience quá lớn tốn thời gian và có thể overfit.

In [ ]:
def patience_experiment(patiences=(5, 10, 20, 30), max_epochs=80):
    rows = []
    histories = {}
    for patience in patiences:
        print(f"\n--- LSTM | patience={patience} ---")
        set_seed(SEED)
        model = RecurrentLotteryModel(model_type="LSTM", hidden_size=96, num_layers=2, dropout=0.25).to(device)
        model, history = train_model(
            model, train_loader, val_loader,
            epochs=max_epochs, lr=1e-3, weight_decay=1e-4, grad_clip=1.0, patience=patience,
        )
        histories[patience] = history
        rows.append({
            "patience": patience,
            "epochs_ran": len(history["val_loss"]),
            "best_val_loss": min(history["val_loss"]),
            "final_val_loss": history["val_loss"][-1],
        })
    return histories, pd.DataFrame(rows)

patience_histories, patience_table = patience_experiment(max_epochs=60)
display(patience_table)

fig, ax = plt.subplots(figsize=(10, 5))
for patience, history in patience_histories.items():
    ax.plot(history["val_loss"], label=f"patience={patience}")
ax.set_title("Val Loss theo patience của Early Stopping")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét cần điền sau khi chạy:** chọn patience dựa trên `best_val_loss` và `epochs_ran`. Với dữ liệu nhiễu, patience vừa phải như 10 hoặc 20 thường hợp lý hơn 5; patience 30 có thể chạy lâu mà không cải thiện nhiều.

## 17. Bảng tổng hợp kết quả thí nghiệm

Bảng dưới gom kết quả chính và có thể dùng trực tiếp trong báo cáo. Sau khi chạy toàn bộ notebook, hãy đối chiếu các bảng phía trên để viết nhận xét cuối cùng.

In [ ]:
summary_rows = []

summary_rows.append({
    "Thí nghiệm": "So sánh mô hình chính",
    "Kết quả chính": f"Best theo avg_hits@6: {results_df.sort_values('avg_hits@6', ascending=False).iloc[0]['model']}",
    "Nhận xét": "So sánh LSTM, GRU và baseline tần suất trên tập test."
})

try:
    best_clip = min(clip_results, key=lambda k: clip_results[k]["val_loss"][-1])
    summary_rows.append({"Thí nghiệm": "Gradient clipping", "Kết quả chính": best_clip, "Nhận xét": "Giúp hạn chế gradient norm khi learning rate lớn."})
except Exception:
    pass

try:
    best_dropout = min(dropout_results, key=lambda k: dropout_results[k]["val_loss"][-1])
    summary_rows.append({"Thí nghiệm": "Dropout", "Kết quả chính": f"dropout={best_dropout}", "Nhận xét": "Giảm overfitting nếu val loss tốt và train-val gap hợp lý."})
except Exception:
    pass

try:
    best_wd = wd_table.sort_values("final_val_loss").iloc[0]["weight_decay"]
    summary_rows.append({"Thí nghiệm": "Weight decay", "Kết quả chính": f"weight_decay={best_wd}", "Nhận xét": "Weight decay quá lớn có thể gây underfitting."})
except Exception:
    pass

try:
    best_init = init_table.sort_values("best_val_loss").iloc[0]["init_method"]
    summary_rows.append({"Thí nghiệm": "Weight initialization", "Kết quả chính": best_init, "Nhận xét": "Zeros thường kém vì symmetry problem; orthogonal thường phù hợp RNN."})
except Exception:
    pass

try:
    best_patience = patience_table.sort_values("best_val_loss").iloc[0]["patience"]
    summary_rows.append({"Thí nghiệm": "Early stopping", "Kết quả chính": f"patience={best_patience}", "Nhận xét": "Patience quá nhỏ dừng sớm; quá lớn tốn thời gian."})
except Exception:
    pass

summary_experiments_df = pd.DataFrame(summary_rows)
display(summary_experiments_df)

## 18. Nhận xét và kết luận mẫu


In [ ]:
best_row = results_df.sort_values("avg_hits@6", ascending=False).iloc[0]
print("KẾT LUẬN")
print("=" * 80)
print(f"Mô hình/baseline có avg_hits@6 cao nhất trên test: {best_row['model']}")
print(f"avg_hits@6 = {best_row['avg_hits@6']:.3f}, exact_match_rate = {best_row['exact_match_rate']:.6f}")
print()
print("Nhận xét:")
print("1. Dữ liệu được xử lý theo đúng thứ tự thời gian, không shuffle để tránh data leakage.")
print("2. LSTM và GRU được huấn luyện như bài toán multi-label 55 nhãn, phù hợp hơn so với hồi quy 6 cột số rời rạc.")
print("3. Vì xổ số có tính ngẫu nhiên rất cao, exact match thường gần 0; avg_hits@6 là metric thực tế hơn để so sánh mô hình.")
print("4. Nếu mô hình không vượt baseline tần suất, đây là kết quả hợp lý và cho thấy dữ liệu lịch sử không chứa quy luật dự đoán mạnh.")
print("5. Có thể cải thiện báo cáo bằng cách tăng epoch, thử sequence_length khác, hidden_size khác, dropout, weight decay và early stopping.")

## 19. Gợi ý trả lời câu hỏi trong báo cáo

1. **Tại sao không shuffle?** Vì dữ liệu có thứ tự thời gian. Shuffle có thể làm thông tin tương lai lọt vào train/validation, khiến đánh giá không còn khách quan.
2. **LSTM và GRU khác nhau thế nào?** LSTM có input/forget/output gate và cell state nên nhiều tham số hơn; GRU dùng update/reset gate nên gọn hơn và thường train nhanh hơn.
3. **Vì sao dùng BCEWithLogitsLoss?** Mỗi kỳ quay có nhiều nhãn đúng cùng lúc trong 55 số, nên đây là bài toán multi-label.
4. **Kết quả dự đoán xổ số có đáng tin không?** Không nên dùng như dự đoán chắc chắn. Mục tiêu của bài là thực hành pipeline deep learning cho dữ liệu chuỗi.